# Data Integration and Filtering for Movie Datasets

This notebook:
- loads the two movie datasets
- matches common movies using normalized `title + release_date` with `id` as a fallback key
- combines common movies into one record and appends movies unique to either source
- checks the `revenue` and `budget` columns after the merge
- removes rows where `revenue` or `budget` is missing or equal to `0`
- enriches the cleaned dataset with `actor_1_popularity`, `actor_2_popularity`, `actor_3_popularity`, and `mpaa_rating`
- exports only the mismatch details text report and the cleaned dataset

## 1. Import libraries and define all input and output paths

This block imports the libraries used in the notebook and sets the file paths for the two source datasets, the mismatch details report, and the final cleaned dataset.

In [1]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import os
import pandas as pd
import requests
import time
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

PROJECT_DIR = Path.cwd()
DATASET_1_PATH = PROJECT_DIR / 'Dataset' / 'movies.csv'
DATASET_2_PATH = PROJECT_DIR / 'Dataset' / 'TMDB_movie_dataset_v11.csv'

OUTPUT_DIR = PROJECT_DIR / 'Dataset' / 'comparison_outputs'
MISMATCH_DETAILS_PATH = OUTPUT_DIR / 'merged_movies_mismatch_details.text'
CLEANED_OUTPUT_PATH = OUTPUT_DIR / 'Movies-Dataset.csv'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Dataset 1: {DATASET_1_PATH}')
print(f'Dataset 2: {DATASET_2_PATH}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Cleaned output: {CLEANED_OUTPUT_PATH}')
print(f'Mismatch details: {MISMATCH_DETAILS_PATH}')

Dataset 1: Dataset\movies.csv
Dataset 2: Dataset\TMDB_movie_dataset_v11.csv
Output dir: Dataset\comparison_outputs
Cleaned output: Dataset\comparison_outputs\Movies-Dataset.csv
Mismatch details: Dataset\comparison_outputs\merged_movies_mismatch_details.text


In [2]:
movie_name = "Meg 2: The Trench"

df1 = pd.read_csv(DATASET_1_PATH)
df2 = pd.read_csv(DATASET_2_PATH)

res1 = df1[df1['title'] == movie_name]
res2 = df2[df2['title'] == movie_name]

print("Dataset 1 match:")
display(res1)

print("\nDataset 2 match:")
display(res2)

Dataset 1 match:


,id,title,genres,original_language,overview,popularity,production_companies,release_date,budget,revenue,runtime,status,tagline,vote_average,vote_count,credits,keywords,poster_path,backdrop_path,recommendations
0,615656,Meg 2: The Trench,Action-Science Fiction-Horror,en,An exploratory dive into the deepest depths of...,8763.998,Apelles Entertainment-Warner Bros. Pictures-di...,2023-08-02,129000000.0,352056482.0,116.0,Released,Back for seconds.,7.079,1365.0,Jason Statham-Wu Jing-Shuya Sophia Cai-Sergio ...,based on novel or book-sequel-kaiju,/4m1Au3YkjqsxF8iwQy0fPYSxE0h.jpg,/qlxy8yo5bcgUw2KAmmojUKp4rHd.jpg,1006462-298618-569094-1061181-346698-1076487-6...



Dataset 2 match:


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,budget,homepage,imdb_id,original_language,original_title,overview,popularity,poster_path,tagline,genres,production_companies,production_countries,spoken_languages,keywords
2129,615656,Meg 2: The Trench,6.912,2034,Released,2023-08-02,384056482,116,False,/5mzr6JZbrqnqD8rCEvPhuCE5Fw2.jpg,129000000,https://www.themeg.movie,tt9224104,en,Meg 2: The Trench,An exploratory dive into the deepest depths of...,1567.273,/4m1Au3YkjqsxF8iwQy0fPYSxE0h.jpg,Back for seconds.,"Action, Science Fiction, Horror","Apelles Entertainment, Warner Bros. Pictures, ...","China, United States of America",English,"based on novel or book, sequel, shark, kaiju, ..."


## 2. Define helper functions for matching, normalization, and mismatch reporting

This block prepares the reusable functions that normalize text and dates, build stable match keys, preserve column order, and record value mismatches for overlapping movies.

In [3]:
def normalize_text_series(series: pd.Series) -> pd.Series:
    return (
        series.fillna('')
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r'\s+', ' ', regex=True)
    )


def normalize_date_series(series: pd.Series) -> pd.Series:
    parsed = pd.to_datetime(series, errors='coerce')
    return parsed.dt.strftime('%Y-%m-%d').fillna('')


def safe_id_series(df: pd.DataFrame, col: str = 'id') -> pd.Series:
    if col not in df.columns:
        return pd.Series(pd.NA, index=df.index, dtype='object')

    raw = df[col].astype(str).str.strip()
    numeric = pd.to_numeric(df[col], errors='coerce')

    out = raw.copy()
    numeric_mask = numeric.notna()
    out.loc[numeric_mask] = numeric.loc[numeric_mask].astype('Int64').astype(str)

    return out.replace({'': pd.NA, 'nan': pd.NA, '<NA>': pd.NA})


def build_match_key(df: pd.DataFrame) -> pd.Series:
    title_key = normalize_text_series(df['title']) if 'title' in df.columns else pd.Series('', index=df.index, dtype='object')
    release_key = normalize_date_series(df['release_date']) if 'release_date' in df.columns else pd.Series('', index=df.index, dtype='object')
    id_key = safe_id_series(df, 'id').fillna('')

    match_key = title_key + '|' + release_key
    empty_title_date = (title_key == '') & (release_key == '')
    match_key = match_key.where(~empty_title_date, 'id|' + id_key)

    return match_key.replace({'|': pd.NA, 'id|': pd.NA})


def normalize_for_compare(series: pd.Series, column_name: str) -> pd.Series:
    col = column_name.lower()
    if 'date' in col:
        return normalize_date_series(series)

    numeric = pd.to_numeric(series, errors='coerce')
    numeric_ratio = numeric.notna().mean() if len(series) else 0

    if numeric_ratio >= 0.9:
        return numeric.round(6).astype(str).replace('nan', '')

    return normalize_text_series(series)


def ordered_union_columns(df_left: pd.DataFrame, df_right: pd.DataFrame) -> list[str]:
    ordered = [col for col in df_left.columns if not col.startswith('_')]
    for col in df_right.columns:
        if not col.startswith('_') and col not in ordered:
            ordered.append(col)
    return ordered


def find_mismatch_details(common_merged: pd.DataFrame, shared_columns: list[str]) -> pd.DataFrame:
    rows = []

    base_columns = [
        col for col in [
            '_merge_key',
            'id_src1', 'id_src2',
            'title_src1', 'title_src2',
            'release_date_src1', 'release_date_src2',
        ]
        if col in common_merged.columns
    ]

    for col in shared_columns:
        left_col = f'{col}_src1'
        right_col = f'{col}_src2'

        if left_col not in common_merged.columns or right_col not in common_merged.columns:
            continue

        left_norm = normalize_for_compare(common_merged[left_col], col)
        right_norm = normalize_for_compare(common_merged[right_col], col)

        comparable = (left_norm != '') & (right_norm != '')
        mismatch_mask = comparable & (left_norm != right_norm)

        if not mismatch_mask.any():
            continue

        context_subset = common_merged.loc[mismatch_mask, base_columns].copy()
        context_subset = context_subset.rename(columns={'_merge_key': 'merge_key'})
        context_subset['column'] = col
        context_subset['value_dataset_1'] = common_merged.loc[mismatch_mask, left_col].to_numpy()
        context_subset['value_dataset_2'] = common_merged.loc[mismatch_mask, right_col].to_numpy()
        rows.append(context_subset)

    if not rows:
        return pd.DataFrame(columns=[
            'merge_key',
            'id_src1', 'id_src2',
            'title_src1', 'title_src2',
            'release_date_src1', 'release_date_src2',
            'column',
            'value_dataset_1',
            'value_dataset_2',
        ])

    return pd.concat(rows, ignore_index=True)

## 3. Load both source datasets and build the merge keys

This block reads the CSV files, creates the normalized match keys, falls back to row-level keys when needed, removes duplicate merge keys inside each dataset, and reports how many rows remain after deduplication.

In [4]:
df1 = pd.read_csv(DATASET_1_PATH, low_memory=False)
df2 = pd.read_csv(DATASET_2_PATH, low_memory=False, encoding='utf-8-sig')

df1['_match_key'] = build_match_key(df1)
df2['_match_key'] = build_match_key(df2)

fallback_merge_key_1 = pd.Series(df1.index.astype(str), index=df1.index).radd('dataset_1_row|')
fallback_merge_key_2 = pd.Series(df2.index.astype(str), index=df2.index).radd('dataset_2_row|')

df1['_merge_key'] = df1['_match_key'].fillna(fallback_merge_key_1)
df2['_merge_key'] = df2['_match_key'].fillna(fallback_merge_key_2)

df1_unique = df1.drop_duplicates('_merge_key', keep='first').copy()
df2_unique = df2.drop_duplicates('_merge_key', keep='first').copy()

load_summary = pd.DataFrame({
    'dataset': ['dataset_1', 'dataset_2'],
    'raw_rows': [len(df1), len(df2)],
    'deduplicated_rows': [len(df1_unique), len(df2_unique)],
    'duplicate_rows_removed': [len(df1) - len(df1_unique), len(df2) - len(df2_unique)],
    'columns': [df1.shape[1] - 2, df2.shape[1] - 2],
})

display(load_summary)

,dataset,raw_rows,deduplicated_rows,duplicate_rows_removed,columns
0,dataset_1,722317,659971,62346,20
1,dataset_2,1351251,1322534,28717,24


In [5]:
movie_name = "Meg 2: The Trench"

d1_movie = df1_unique[df1_unique['title'].astype(str).str.strip().eq(movie_name)].copy()
d2_movie = df2_unique[df2_unique['title'].astype(str).str.strip().eq(movie_name)].copy()

print("Dataset 1 matches:")
display(d1_movie)

print("Dataset 2 matches:")
display(d2_movie)

Dataset 1 matches:


,id,title,genres,original_language,overview,popularity,production_companies,release_date,budget,revenue,runtime,status,tagline,vote_average,vote_count,credits,keywords,poster_path,backdrop_path,recommendations,_match_key,_merge_key
0,615656,Meg 2: The Trench,Action-Science Fiction-Horror,en,An exploratory dive into the deepest depths of...,8763.998,Apelles Entertainment-Warner Bros. Pictures-di...,2023-08-02,129000000.0,352056482.0,116.0,Released,Back for seconds.,7.079,1365.0,Jason Statham-Wu Jing-Shuya Sophia Cai-Sergio ...,based on novel or book-sequel-kaiju,/4m1Au3YkjqsxF8iwQy0fPYSxE0h.jpg,/qlxy8yo5bcgUw2KAmmojUKp4rHd.jpg,1006462-298618-569094-1061181-346698-1076487-6...,meg 2: the trench|2023-08-02,meg 2: the trench|2023-08-02


Dataset 2 matches:


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,budget,homepage,imdb_id,original_language,original_title,overview,popularity,poster_path,tagline,genres,production_companies,production_countries,spoken_languages,keywords,_match_key,_merge_key
2129,615656,Meg 2: The Trench,6.912,2034,Released,2023-08-02,384056482,116,False,/5mzr6JZbrqnqD8rCEvPhuCE5Fw2.jpg,129000000,https://www.themeg.movie,tt9224104,en,Meg 2: The Trench,An exploratory dive into the deepest depths of...,1567.273,/4m1Au3YkjqsxF8iwQy0fPYSxE0h.jpg,Back for seconds.,"Action, Science Fiction, Horror","Apelles Entertainment, Warner Bros. Pictures, ...","China, United States of America",English,"based on novel or book, sequel, shark, kaiju, ...",meg 2: the trench|2023-08-02,meg 2: the trench|2023-08-02


## 4. Separate common movies from source-specific movies and inspect mismatches

This block finds the movies shared by both datasets, isolates the movies unique to each source, and summarizes any field-level mismatches across shared columns for the common matches.

In [6]:
common_keys = set(df1_unique['_merge_key']).intersection(set(df2_unique['_merge_key']))

common_src1 = df1_unique[df1_unique['_merge_key'].isin(common_keys)].copy()
common_src2 = df2_unique[df2_unique['_merge_key'].isin(common_keys)].copy()

unique_src1 = df1_unique[~df1_unique['_merge_key'].isin(common_keys)].copy()
unique_src2 = df2_unique[~df2_unique['_merge_key'].isin(common_keys)].copy()

common_merged = common_src1.merge(
    common_src2,
    on='_merge_key',
    how='inner',
    suffixes=('_src1', '_src2'),
    validate='one_to_one',
)

shared_columns = sorted(
    (set(df1_unique.columns) & set(df2_unique.columns))
    - {'_match_key', '_merge_key'}
)

mismatch_details = find_mismatch_details(common_merged, shared_columns)

if mismatch_details.empty:
    mismatch_summary = pd.DataFrame(columns=['column', 'mismatch_rows'])
else:
    mismatch_summary = (
        mismatch_details['column']
        .value_counts()
        .rename_axis('column')
        .reset_index(name='mismatch_rows')
    )

merge_breakdown = pd.DataFrame({
    'metric': [
        'common_movies_overlapped',
        'unique_movies_dataset_1',
        'unique_movies_dataset_2',
    ],
    'value': [
        len(common_merged),
        len(unique_src1),
        len(unique_src2),
    ],
})

display(merge_breakdown)
display(mismatch_summary.head(20))

,metric,value
0,common_movies_overlapped,632990
1,unique_movies_dataset_1,26981
2,unique_movies_dataset_2,689544


,column,mismatch_rows
0,budget,632990
1,revenue,632990
2,runtime,632990
3,vote_count,632990
4,overview,380604
5,popularity,359895
6,genres,173881
7,keywords,119660
8,vote_average,118188
9,production_companies,97160


In [7]:
movie_name = "Meg 2: The Trench"

# 1. Check whether the movie exists in each deduplicated source
movie_src1 = df1_unique[df1_unique['title'].astype(str).str.strip() == movie_name].copy()
movie_src2 = df2_unique[df2_unique['title'].astype(str).str.strip() == movie_name].copy()

print("Dataset 1 unique row(s):")
display(movie_src1)

print("Dataset 2 unique row(s):")
display(movie_src2)

# 2. Check if both sides got the same merge key
print("Dataset 1 merge keys:")
display(movie_src1[['title', '_match_key', '_merge_key']])

print("Dataset 2 merge keys:")
display(movie_src2[['title', '_match_key', '_merge_key']])

movie_keys = set(movie_src1['_merge_key']).intersection(set(movie_src2['_merge_key']))
print("Common merge key(s):", movie_keys)

# 3. Check if it actually appears in common_merged
movie_merged = common_merged[common_merged['_merge_key'].isin(movie_keys)].copy()

print("Merged row(s):")
display(movie_merged)

Dataset 1 unique row(s):


,id,title,genres,original_language,overview,popularity,production_companies,release_date,budget,revenue,runtime,status,tagline,vote_average,vote_count,credits,keywords,poster_path,backdrop_path,recommendations,_match_key,_merge_key
0,615656,Meg 2: The Trench,Action-Science Fiction-Horror,en,An exploratory dive into the deepest depths of...,8763.998,Apelles Entertainment-Warner Bros. Pictures-di...,2023-08-02,129000000.0,352056482.0,116.0,Released,Back for seconds.,7.079,1365.0,Jason Statham-Wu Jing-Shuya Sophia Cai-Sergio ...,based on novel or book-sequel-kaiju,/4m1Au3YkjqsxF8iwQy0fPYSxE0h.jpg,/qlxy8yo5bcgUw2KAmmojUKp4rHd.jpg,1006462-298618-569094-1061181-346698-1076487-6...,meg 2: the trench|2023-08-02,meg 2: the trench|2023-08-02


Dataset 2 unique row(s):


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,budget,homepage,imdb_id,original_language,original_title,overview,popularity,poster_path,tagline,genres,production_companies,production_countries,spoken_languages,keywords,_match_key,_merge_key
2129,615656,Meg 2: The Trench,6.912,2034,Released,2023-08-02,384056482,116,False,/5mzr6JZbrqnqD8rCEvPhuCE5Fw2.jpg,129000000,https://www.themeg.movie,tt9224104,en,Meg 2: The Trench,An exploratory dive into the deepest depths of...,1567.273,/4m1Au3YkjqsxF8iwQy0fPYSxE0h.jpg,Back for seconds.,"Action, Science Fiction, Horror","Apelles Entertainment, Warner Bros. Pictures, ...","China, United States of America",English,"based on novel or book, sequel, shark, kaiju, ...",meg 2: the trench|2023-08-02,meg 2: the trench|2023-08-02


Dataset 1 merge keys:


,title,_match_key,_merge_key
0,Meg 2: The Trench,meg 2: the trench|2023-08-02,meg 2: the trench|2023-08-02


Dataset 2 merge keys:


,title,_match_key,_merge_key
2129,Meg 2: The Trench,meg 2: the trench|2023-08-02,meg 2: the trench|2023-08-02


Common merge key(s): {'meg 2: the trench|2023-08-02'}
Merged row(s):


,id_src1,title_src1,genres_src1,original_language_src1,overview_src1,popularity_src1,production_companies_src1,release_date_src1,budget_src1,revenue_src1,runtime_src1,status_src1,tagline_src1,vote_average_src1,vote_count_src1,credits,keywords_src1,poster_path_src1,backdrop_path_src1,recommendations,_match_key_src1,_merge_key,id_src2,title_src2,vote_average_src2,vote_count_src2,status_src2,release_date_src2,revenue_src2,runtime_src2,adult,backdrop_path_src2,budget_src2,homepage,imdb_id,original_language_src2,original_title,overview_src2,popularity_src2,poster_path_src2,tagline_src2,genres_src2,production_companies_src2,production_countries,spoken_languages,keywords_src2,_match_key_src2
0,615656,Meg 2: The Trench,Action-Science Fiction-Horror,en,An exploratory dive into the deepest depths of...,8763.998,Apelles Entertainment-Warner Bros. Pictures-di...,2023-08-02,129000000.0,352056482.0,116.0,Released,Back for seconds.,7.079,1365.0,Jason Statham-Wu Jing-Shuya Sophia Cai-Sergio ...,based on novel or book-sequel-kaiju,/4m1Au3YkjqsxF8iwQy0fPYSxE0h.jpg,/qlxy8yo5bcgUw2KAmmojUKp4rHd.jpg,1006462-298618-569094-1061181-346698-1076487-6...,meg 2: the trench|2023-08-02,meg 2: the trench|2023-08-02,615656,Meg 2: The Trench,6.912,2034,Released,2023-08-02,384056482,116,False,/5mzr6JZbrqnqD8rCEvPhuCE5Fw2.jpg,129000000,https://www.themeg.movie,tt9224104,en,Meg 2: The Trench,An exploratory dive into the deepest depths of...,1567.273,/4m1Au3YkjqsxF8iwQy0fPYSxE0h.jpg,Back for seconds.,"Action, Science Fiction, Horror","Apelles Entertainment, Warner Bros. Pictures, ...","China, United States of America",English,"based on novel or book, sequel, shark, kaiju, ...",meg 2: the trench|2023-08-02


## 5. Build the merged dataset and export the mismatch details report

This block overlaps the matching movies into single records, appends the movies that appear in only one source, keeps the merged dataset in memory, and writes the mismatch details report to disk.

In [8]:
final_columns = ordered_union_columns(df1_unique, df2_unique)

common_combined = pd.DataFrame({
    'record_origin': 'common_movie',
    'merge_key': common_merged['_merge_key'],
})

for col in final_columns:
    left_col = f'{col}_src1'
    right_col = f'{col}_src2'

    if left_col in common_merged.columns and right_col in common_merged.columns:
        common_combined[col] = (
            common_merged[left_col]
            .replace('', pd.NA)
            .combine_first(common_merged[right_col].replace('', pd.NA))
        )
    elif left_col in common_merged.columns:
        common_combined[col] = common_merged[left_col].replace('', pd.NA)
    elif right_col in common_merged.columns:
        common_combined[col] = common_merged[right_col].replace('', pd.NA)
    elif col in common_merged.columns:
        common_combined[col] = common_merged[col].replace('', pd.NA)
    else:
        common_combined[col] = pd.NA

unique_src1_export = unique_src1.reindex(columns=final_columns).copy()
unique_src1_export.insert(0, 'merge_key', unique_src1['_merge_key'])
unique_src1_export.insert(0, 'record_origin', 'dataset_1_only')

unique_src2_export = unique_src2.reindex(columns=final_columns).copy()
unique_src2_export.insert(0, 'merge_key', unique_src2['_merge_key'])
unique_src2_export.insert(0, 'record_origin', 'dataset_2_only')

final_merged_movies = pd.concat(
    [common_combined, unique_src1_export, unique_src2_export],
    ignore_index=True,
    sort=False,
)

mismatch_details.to_csv(MISMATCH_DETAILS_PATH, index=False, encoding='utf-8-sig')

display(final_merged_movies.head())

,record_origin,merge_key,id,title,genres,original_language,overview,popularity,production_companies,release_date,budget,revenue,runtime,status,tagline,vote_average,vote_count,credits,keywords,poster_path,backdrop_path,recommendations,adult,homepage,imdb_id,original_title,production_countries,spoken_languages
0,common_movie,meg 2: the trench|2023-08-02,615656,Meg 2: The Trench,Action-Science Fiction-Horror,en,An exploratory dive into the deepest depths of...,8763.998,Apelles Entertainment-Warner Bros. Pictures-di...,2023-08-02,129000000.0,352056482.0,116.0,Released,Back for seconds.,7.079,1365.0,Jason Statham-Wu Jing-Shuya Sophia Cai-Sergio ...,based on novel or book-sequel-kaiju,/4m1Au3YkjqsxF8iwQy0fPYSxE0h.jpg,/qlxy8yo5bcgUw2KAmmojUKp4rHd.jpg,1006462-298618-569094-1061181-346698-1076487-6...,False,https://www.themeg.movie,tt9224104,Meg 2: The Trench,"China, United States of America",English
1,common_movie,the pope's exorcist|2023-04-05,758323,The Pope's Exorcist,Horror-Mystery-Thriller,en,Father Gabriele Amorth Chief Exorcist of the V...,5953.227,Screen Gems-2.0 Entertainment-Jesus & Mary-Wor...,2023-04-05,18000000.0,65675816.0,103.0,Released,Inspired by the actual files of Father Gabriel...,7.433,545.0,Russell Crowe-Daniel Zovatto-Alex Essoe-Franco...,spain-rome italy-vatican-pope-pig-possession-c...,/9JBEPLTPSm0d1mbEcLxULjJq9Eh.jpg,/hiHGRbyTcbZoLsYYkO4QiCLYe34.jpg,713704-296271-502356-1076605-1084225-1008005-9...,False,https://www.thepopes-exorcist.movie,tt13375076,The Pope's Exorcist,"Spain, United Kingdom, United States of America","English, Fulah, Spanish, Latin, German, Italian"
2,common_movie,transformers: rise of the beasts|2023-06-06,667538,Transformers: Rise of the Beasts,Action-Adventure-Science Fiction,en,When a new threat capable of destroying the en...,5409.104,Skydance-Paramount-di Bonaventura Pictures-Bay...,2023-06-06,200000000.0,407045464.0,127.0,Released,Unite or fall.,7.340,1007.0,Anthony Ramos-Dominique Fishback-Luna Lauren V...,peru-alien-end of the world-based on cartoon-b...,/gPbM0MK8CP8A174rmUwGsADNYKD.jpg,/woJbg7ZqidhpvqFGGMRhWQNoxwa.jpg,496450-569094-298618-385687-877100-598331-4628...,False,https://www.transformersmovie.com,tt5090568,Transformers: Rise of the Beasts,United States of America,"Quechua, Spanish, English"
3,common_movie,ant-man and the wasp: quantumania|2023-02-15,640146,Ant-Man and the Wasp: Quantumania,Action-Adventure-Science Fiction,en,Super-Hero partners Scott Lang and Hope van Dy...,4425.387,Marvel Studios-Kevin Feige Productions,2023-02-15,200000000.0,475766228.0,125.0,Released,Witness the beginning of a new dynasty.,6.507,2811.0,Paul Rudd-Evangeline Lilly-Jonathan Majors-Kat...,hero-ant-sequel-superhero-based on comic-famil...,/qnqGbB22YJ7dSs4o6M7exTpNxPz.jpg,/m8JTwHFwX7I7JY5fPe4SjqejWag.jpg,823999-676841-868759-734048-267805-965839-1033...,False,https://www.marvel.com/movies/ant-man-and-the-...,tt10954600,Ant-Man and the Wasp: Quantumania,United States of America,English
4,common_movie,creed iii|2023-03-01,677179,Creed III,Drama-Action,en,After dominating the boxing world Adonis Creed...,3994.342,Metro-Goldwyn-Mayer-Proximity Media-Balboa Pro...,2023-03-01,75000000.0,269000000.0,116.0,Released,You can't run from your past.,7.262,1129.0,Michael B. Jordan-Tessa Thompson-Jonathan Majo...,philadelphia pennsylvania-husband wife relatio...,/cvsXj3I9Q2iyyIo95AecSd1tad7.jpg,/5i6SjyDbDWqyun8klUuCxrlFbyw.jpg,965839-267805-943822-842942-1035806-823999-107...,False,https://www.mgmstudios.com/creed-iii,tt11145118,Creed III,United States of America,"English, Spanish"


In [9]:
print("credits in df1_unique:", 'credits' in df1_unique.columns)
print("credits in df2_unique:", 'credits' in df2_unique.columns)

print("credits_src1 in common_merged:", 'credits_src1' in common_merged.columns)
print("credits_src2 in common_merged:", 'credits_src2' in common_merged.columns)

credits in df1_unique: True
credits in df2_unique: False
credits_src1 in common_merged: False
credits_src2 in common_merged: False


## 6. Inspect the `revenue` and `budget` columns before filtering

This block counts how many `revenue` and `budget` values are negative, equal to `0`, positive, or missing in the merged dataset before the final cleaning step.

In [10]:
cols = ['revenue', 'budget']
missing_required = [col for col in cols if col not in final_merged_movies.columns]

if missing_required:
    raise KeyError(f'Missing required columns in merged dataset: {missing_required}')

numeric_df = final_merged_movies[cols].apply(pd.to_numeric, errors='coerce')

count_summary = pd.DataFrame({
    'less_than_0': (numeric_df < 0).sum(),
    'equal_to_0': (numeric_df == 0).sum(),
    'greater_than_0': (numeric_df > 0).sum(),
    'missing': numeric_df.isna().sum(),
})

display(count_summary)

,less_than_0,equal_to_0,greater_than_0,missing
revenue,1,1326310,23204,0
budget,0,1275849,73666,0


## 7. Remove rows where `revenue` or `budget` is missing or equal to `0`

This block converts `revenue` and `budget` to numeric values, keeps only the rows where both columns are present and non-zero, and exports the final cleaned dataset.

In [11]:
cleaned_movies = final_merged_movies.copy()
cleaned_movies['revenue'] = pd.to_numeric(cleaned_movies['revenue'], errors='coerce')
cleaned_movies['budget'] = pd.to_numeric(cleaned_movies['budget'], errors='coerce')

raw_revenue = pd.to_numeric(final_merged_movies['revenue'], errors='coerce')
raw_budget = pd.to_numeric(final_merged_movies['budget'], errors='coerce')

valid_revenue_mask = cleaned_movies['revenue'].notna() & (cleaned_movies['revenue'] != 0)
valid_budget_mask = cleaned_movies['budget'].notna() & (cleaned_movies['budget'] != 0)

cleaned_movies = cleaned_movies[valid_revenue_mask & valid_budget_mask].copy()

filter_summary = pd.DataFrame({
    'metric': [
        'rows_in_raw_merged_dataset',
        'rows_with_missing_revenue',
        'rows_with_missing_budget',
        'rows_with_revenue_equal_to_0',
        'rows_with_budget_equal_to_0',
        'rows_after_filtering',
        'rows_removed_by_filter',
    ],
    'value': [
        len(final_merged_movies),
        int(raw_revenue.isna().sum()),
        int(raw_budget.isna().sum()),
        int((raw_revenue == 0).sum()),
        int((raw_budget == 0).sum()),
        len(cleaned_movies),
        len(final_merged_movies) - len(cleaned_movies),
    ],
})

display(filter_summary)
display(cleaned_movies.head())


,metric,value
0,rows_in_raw_merged_dataset,1349515
1,rows_with_missing_revenue,0
2,rows_with_missing_budget,0
3,rows_with_revenue_equal_to_0,1326310
4,rows_with_budget_equal_to_0,1275849
5,rows_after_filtering,16441
6,rows_removed_by_filter,1333074


,record_origin,merge_key,id,title,genres,original_language,overview,popularity,production_companies,release_date,budget,revenue,runtime,status,tagline,vote_average,vote_count,credits,keywords,poster_path,backdrop_path,recommendations,adult,homepage,imdb_id,original_title,production_countries,spoken_languages
0,common_movie,meg 2: the trench|2023-08-02,615656,Meg 2: The Trench,Action-Science Fiction-Horror,en,An exploratory dive into the deepest depths of...,8763.998,Apelles Entertainment-Warner Bros. Pictures-di...,2023-08-02,129000000.0,352056482.0,116.0,Released,Back for seconds.,7.079,1365.0,Jason Statham-Wu Jing-Shuya Sophia Cai-Sergio ...,based on novel or book-sequel-kaiju,/4m1Au3YkjqsxF8iwQy0fPYSxE0h.jpg,/qlxy8yo5bcgUw2KAmmojUKp4rHd.jpg,1006462-298618-569094-1061181-346698-1076487-6...,False,https://www.themeg.movie,tt9224104,Meg 2: The Trench,"China, United States of America",English
1,common_movie,the pope's exorcist|2023-04-05,758323,The Pope's Exorcist,Horror-Mystery-Thriller,en,Father Gabriele Amorth Chief Exorcist of the V...,5953.227,Screen Gems-2.0 Entertainment-Jesus & Mary-Wor...,2023-04-05,18000000.0,65675816.0,103.0,Released,Inspired by the actual files of Father Gabriel...,7.433,545.0,Russell Crowe-Daniel Zovatto-Alex Essoe-Franco...,spain-rome italy-vatican-pope-pig-possession-c...,/9JBEPLTPSm0d1mbEcLxULjJq9Eh.jpg,/hiHGRbyTcbZoLsYYkO4QiCLYe34.jpg,713704-296271-502356-1076605-1084225-1008005-9...,False,https://www.thepopes-exorcist.movie,tt13375076,The Pope's Exorcist,"Spain, United Kingdom, United States of America","English, Fulah, Spanish, Latin, German, Italian"
2,common_movie,transformers: rise of the beasts|2023-06-06,667538,Transformers: Rise of the Beasts,Action-Adventure-Science Fiction,en,When a new threat capable of destroying the en...,5409.104,Skydance-Paramount-di Bonaventura Pictures-Bay...,2023-06-06,200000000.0,407045464.0,127.0,Released,Unite or fall.,7.340,1007.0,Anthony Ramos-Dominique Fishback-Luna Lauren V...,peru-alien-end of the world-based on cartoon-b...,/gPbM0MK8CP8A174rmUwGsADNYKD.jpg,/woJbg7ZqidhpvqFGGMRhWQNoxwa.jpg,496450-569094-298618-385687-877100-598331-4628...,False,https://www.transformersmovie.com,tt5090568,Transformers: Rise of the Beasts,United States of America,"Quechua, Spanish, English"
3,common_movie,ant-man and the wasp: quantumania|2023-02-15,640146,Ant-Man and the Wasp: Quantumania,Action-Adventure-Science Fiction,en,Super-Hero partners Scott Lang and Hope van Dy...,4425.387,Marvel Studios-Kevin Feige Productions,2023-02-15,200000000.0,475766228.0,125.0,Released,Witness the beginning of a new dynasty.,6.507,2811.0,Paul Rudd-Evangeline Lilly-Jonathan Majors-Kat...,hero-ant-sequel-superhero-based on comic-famil...,/qnqGbB22YJ7dSs4o6M7exTpNxPz.jpg,/m8JTwHFwX7I7JY5fPe4SjqejWag.jpg,823999-676841-868759-734048-267805-965839-1033...,False,https://www.marvel.com/movies/ant-man-and-the-...,tt10954600,Ant-Man and the Wasp: Quantumania,United States of America,English
4,common_movie,creed iii|2023-03-01,677179,Creed III,Drama-Action,en,After dominating the boxing world Adonis Creed...,3994.342,Metro-Goldwyn-Mayer-Proximity Media-Balboa Pro...,2023-03-01,75000000.0,269000000.0,116.0,Released,You can't run from your past.,7.262,1129.0,Michael B. Jordan-Tessa Thompson-Jonathan Majo...,philadelphia pennsylvania-husband wife relatio...,/cvsXj3I9Q2iyyIo95AecSd1tad7.jpg,/5i6SjyDbDWqyun8klUuCxrlFbyw.jpg,965839-267805-943822-842942-1035806-823999-107...,False,https://www.mgmstudios.com/creed-iii,tt11145118,Creed III,United States of America,"English, Spanish"


## 8. Add parsed actor columns, TMDB actor popularity, and MPAA rating features

This block derives `actor_1`, `actor_2`, and `actor_3` from `credits`, keeps the parsed actor names in the dataset, records the matched TMDB actor name and TMDB person ID for auditing, adds actor popularity features, derives `mpaa_rating` from TMDB release dates, saves the final cleaned dataset, and prints a short output summary with enrichment diagnostics.


In [12]:
display(cleaned_movies.shape[0])

16441

In [13]:
from tmdb_enrichment import enrich_cleaned_movies

print("Before enrichment:")
cleaned_movies, actor_enrichment_summary, output_summary, summary_text = enrich_cleaned_movies(
    cleaned_movies,
    final_merged_movies=final_merged_movies,
    df1=df1,
    df2=df2,
    df1_unique=df1_unique,
    df2_unique=df2_unique,
    common_combined=common_combined,
    unique_src1_export=unique_src1_export,
    unique_src2_export=unique_src2_export,
    mismatch_summary=mismatch_summary,
    mismatch_details_path=MISMATCH_DETAILS_PATH,
    cleaned_output_path=CLEANED_OUTPUT_PATH,
)

print("After enrichment:")

print(summary_text)
display(actor_enrichment_summary)
display(
    cleaned_movies[
        [
            'title',
            'actor_1', 'actor_1_tmdb_name', 'actor_1_popularity',
            'actor_2', 'actor_2_tmdb_name', 'actor_2_popularity',
            'actor_3', 'actor_3_tmdb_name', 'actor_3_popularity',
            'mpaa_rating',
        ]
    ].head()
)
display(output_summary)


Before enrichment:
Inside enrich_cleaned_movies function
Start
Done
Done1


KeyboardInterrupt: 